# Phase 2: Data Preprocessing

Cleans and combines the text fields from the raw job postings, and saves the
processed DataFrame for use in Phase 3 (Feature Engineering).

In [1]:
import pandas as pd
import re

DATA_PATH = "../data/fake_job_postings.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

Loaded 17880 rows, 18 columns


## Text cleaning function

Lowercases text, strips HTML tags, URLs, EMSCAD's anonymized placeholder tokens
(e.g. `#URL_...#`), punctuation, and digits.

In [2]:
def clean_text(text: str) -> str:
    """Lowercase, strip HTML tags, URLs, placeholder tokens, punctuation, and digits noise."""
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)                 # HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # URLs
    text = re.sub(r"#url_\w+#|#email_\w+#|#phone_\w+#", " ", text)  # EMSCAD's anonymized placeholders
    text = re.sub(r"[^a-z\s]", " ", text)               # punctuation/digits
    text = re.sub(r"\s+", " ", text).strip()
    return text

## Fill missing values, combine fields, apply cleaning

Missing text fields are filled with an empty string (rather than dropping rows),
then the five text fields are concatenated into one combined text column, cleaned.

In [3]:
TEXT_FIELDS = ["title", "company_profile", "description", "requirements", "benefits"]
for col in TEXT_FIELDS:
    df[col] = df[col].fillna("")

df["full_text"] = df[TEXT_FIELDS].agg(" ".join, axis=1)
df["full_text_clean"] = df["full_text"].apply(clean_text)

print("Example before cleaning:\n", df["full_text"].iloc[0][:300], "...\n")
print("Example after cleaning:\n", df["full_text_clean"].iloc[0][:300], "...")

Example before cleaning:
 Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and  ...

Example after cleaning:
 marketing intern we re food and we ve created a groundbreaking and award winning cooking site we support connect and celebrate home cooks and give them everything they need in one place we have a top editorial business and engineering team we re focused on using technology to find new and better way ...


## Save processed data for Phase 3

In [4]:
df.to_pickle("../data/processed_postings.pkl")
print("Saved processed data to data/processed_postings.pkl")

Saved processed data to data/processed_postings.pkl
